<a href="https://colab.research.google.com/github/Marion13673/MARION/blob/main/Notebook_flight_on_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**✈️PREDICCIÓN DE RETRASOS DE VUELOS N°2**

#**1. Importando las librerías**

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import os

#**2. Lectura de los datos**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

datos = pd.read_csv("/content/drive/MyDrive/HACKATHON/ARCHIVOS 2015/flight_clean.csv")

datos.head(5)

/tmp/ipython-input-2759207130.py:1: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  datos = pd.read_csv("/content/drive/MyDrive/HACKATHON/ARCHIVOS 2015/flight_clean.csv")


,Unnamed: 0,ANO,MES,DIA,DIA_SEMANA,AEROLINEA,NUMERO_VUELO,NUMERO_DEL_AVION,AEROPUERTO_ORIGEN,AEROPUERTO_DESTINO,...,DESVIADO,CANCELADO,RAZON_CANCELACION,RETRASO_SISTEMA_AEREO,RETRASO_SEGURIDAD,RETRASO_AEROLINEA,RETRASO_AVION_TARDIO,RETRASO_CLIMA,LLEGADA_PROGRAMA,RETRASO_GRAVE
0,0,2015,1,1,4,AS,98,N407AS,ANC,SEA,...,0,0,No Cancelado,0.0,0.0,0.0,0.0,0.0,7.166667,0
1,1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,...,0,0,No Cancelado,0.0,0.0,0.0,0.0,0.0,12.500000,0
2,2,2015,1,1,4,US,840,N171US,SFO,CLT,...,0,0,No Cancelado,0.0,0.0,0.0,0.0,0.0,13.433333,0
3,3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,...,0,0,No Cancelado,0.0,0.0,0.0,0.0,0.0,13.416667,0
4,4,2015,1,1,4,AS,135,N527AS,SEA,ANC,...,0,0,No Cancelado,0.0,0.0,0.0,0.0,0.0,5.333333,0


In [ ]:
!pip install catboost

#**3. Modelo CatBoost de Predicción de Retrasos Graves en Vuelos**

#**📌 Flujo completo**

- Definición del target (RETRASO_GRAVE)
- Se calcula como 1 si la suma de las causas de retraso (RETRASO_TOTAL) es ≥ 30 minutos.
- Es decir, el modelo aprende a distinguir entre vuelos con retraso grave (≥30 min) y vuelos puntuales (<30 min).

**Entrenamiento del modelo**

- El modelo se entrena para predecir la probabilidad de que un vuelo pertenezca a la clase RETRASO_GRAVE = 1.

**Por ejemplo:**
- Si el modelo devuelve 0.6, significa que estima un 60% de probabilidad de retraso ≥30 min.
- Si devuelve 0.9, significa un 90% de probabilidad de retraso ≥30 min.
- Uso del umbral (0.7912)


In [5]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import pandas as pd
import numpy as np

# ---------------------------
# 1. Target: retraso grave >= 30 min
# ---------------------------
causas = ['RETRASO_SISTEMA_AEREO','RETRASO_SEGURIDAD',
          'RETRASO_AEROLINEA','RETRASO_AVION_TARDIO','RETRASO_CLIMA']
datos["RETRASO_TOTAL"] = datos[causas].sum(axis=1)
datos["RETRASO_GRAVE"] = (datos["RETRASO_TOTAL"] >= 30).astype(int)

# ---------------------------
# 2. Feature engineering de partida
# ---------------------------
datos["FECHA_COMPLETA"] = pd.to_datetime(
    datos[["ANO","MES","DIA"]].rename(columns={"ANO":"year","MES":"month","DIA":"day"})
)

datos["DIA_SEMANA"] = datos["FECHA_COMPLETA"].dt.dayofweek
datos["MES_PARTIDA"] = datos["FECHA_COMPLETA"].dt.month
datos["ES_FIN_DE_SEMANA"] = datos["DIA_SEMANA"].isin([5,6]).astype(int)

datos["TEMPORADA"] = pd.cut(datos["MES_PARTIDA"],
                            bins=[0,3,6,9,12],
                            labels=["Verano","Otoño","Invierno","Primavera"],
                            right=True)

# ---------------------------
# 3. Variables de llegada
# ---------------------------
datos["HORA_LLEGADA"] = pd.to_numeric(datos["HORA_LLEGADA"], errors="coerce")
datos["HORA_LLEGADA"] = datos["HORA_LLEGADA"].fillna(datos["HORA_LLEGADA"].mean()).astype(int)

datos["FRANJA_HORARIA_LLEGADA"] = pd.cut(datos["HORA_LLEGADA"],
                                         bins=[0,6,12,18,24],
                                         labels=["Madrugada","Mañana","Tarde","Noche"],
                                         right=False)

# ---------------------------
# 4. Nueva variable: LLEGADA_PROGRAMA
# ---------------------------
datos["LLEGADA_PROGRAMA"] = pd.to_numeric(datos["LLEGADA_PROGRAMA"], errors="coerce")
datos["LLEGADA_PROGRAMA"] = datos["LLEGADA_PROGRAMA"].fillna(datos["LLEGADA_PROGRAMA"].mean()).astype(int)

datos["FRANJA_LLEGADA_PROGRAMA"] = pd.cut(datos["LLEGADA_PROGRAMA"],
                                          bins=[0,6,12,18,24],
                                          labels=["Madrugada","Mañana","Tarde","Noche"],
                                          right=False)

# ---------------------------
# 5. Features finales
# ---------------------------
X = datos[[
    "AEROLINEA","AEROPUERTO_ORIGEN","AEROPUERTO_DESTINO",
    "DISTANCIA","DIA_SEMANA","MES_PARTIDA","ES_FIN_DE_SEMANA",
    "TEMPORADA",
    "HORA_LLEGADA","FRANJA_HORARIA_LLEGADA",
    "LLEGADA_PROGRAMA","FRANJA_LLEGADA_PROGRAMA"
]]
y = datos["RETRASO_GRAVE"]

categorical_cols = [
    "AEROLINEA","AEROPUERTO_ORIGEN","AEROPUERTO_DESTINO",
    "FRANJA_HORARIA_LLEGADA","FRANJA_LLEGADA_PROGRAMA","DIA_SEMANA","TEMPORADA"
]

# Limpieza automática
for col in categorical_cols:
    X.loc[:, col] = X[col].astype(str).fillna("missing")

num_cols = ["DISTANCIA","DIA_SEMANA","MES_PARTIDA","ES_FIN_DE_SEMANA",
            "HORA_LLEGADA","LLEGADA_PROGRAMA"]
for col in num_cols:
    X.loc[:, col] = pd.to_numeric(X[col], errors="coerce")
    X.loc[:, col] = X[col].fillna(X[col].mean())

# ---------------------------
# 6. Train/Test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------
# 7. Modelo CatBoost
# ---------------------------
at_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=5,
    random_seed=42,
    verbose=100,
    class_weights=[1, len(y_train[y_train==0]) / len(y_train[y_train==1])]
)

at_model.fit(
    X_train, y_train,
    cat_features=categorical_cols,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50
)

# ---------------------------
# 8. Evaluación con umbral fijo = 0.7912
# ---------------------------
y_proba = at_model.predict_proba(X_test)[:,1]

umbral_optimo = 0.7912
y_pred_opt = (y_proba >= umbral_optimo).astype(int)

print("Umbral fijo:", umbral_optimo)
print("Precisión:", precision_score(y_test,y_pred_opt))
print("Recall:", recall_score(y_test,y_pred_opt))
print("F1:", f1_score(y_test,y_pred_opt))
print("Matriz de confusión:\n", confusion_matrix(y_test,y_pred_opt))
print("ROC-AUC:", roc_auc_score(y_test,y_proba))

/tmp/ipython-input-797594670.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Mañana' 'Tarde' 'Tarde' ... 'Mañana' 'Madrugada' 'Mañana']' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X.loc[:, col] = X[col].astype(str).fillna("missing")
/tmp/ipython-input-797594670.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Mañana' 'Tarde' 'Tarde' ... 'Mañana' 'Madrugada' 'Mañana']' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X.loc[:, col] = X[col].astype(str).fillna("missing")
/tmp/ipython-input-797594670.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['3' '3' '3' ... '3' '3' '3']' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  X.lo

0:	learn: 0.6836883	test: 0.6837011	best: 0.6837011 (0)	total: 11.4s	remaining: 3h 9m 46s
100:	learn: 0.4350429	test: 0.4361876	best: 0.4361876 (100)	total: 14m 8s	remaining: 2h 5m 48s
200:	learn: 0.3406936	test: 0.3419550	best: 0.3419550 (200)	total: 29m 3s	remaining: 1h 55m 28s
300:	learn: 0.2874767	test: 0.2888278	best: 0.2888278 (300)	total: 44m 32s	remaining: 1h 43m 26s
400:	learn: 0.2612405	test: 0.2626773	best: 0.2626773 (400)	total: 59m 7s	remaining: 1h 28m 19s
500:	learn: 0.2456286	test: 0.2471395	best: 0.2471395 (500)	total: 1h 14m 6s	remaining: 1h 13m 49s
600:	learn: 0.2331676	test: 0.2347514	best: 0.2347514 (600)	total: 1h 28m 56s	remaining: 59m 2s
700:	learn: 0.2266556	test: 0.2282623	best: 0.2282623 (700)	total: 1h 43m 21s	remaining: 44m 5s
800:	learn: 0.2222793	test: 0.2239328	best: 0.2239328 (800)	total: 1h 57m 51s	remaining: 29m 16s
900:	learn: 0.2184137	test: 0.2201540	best: 0.2201540 (900)	total: 2h 12m 23s	remaining: 14m 32s
999:	learn: 0.2157336	test: 0.2175249	bes

In [ ]:
at_model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,HORA_LLEGADA,51.613366
1,LLEGADA_PROGRAMA,40.374843
2,FRANJA_HORARIA_LLEGADA,4.238255
3,FRANJA_LLEGADA_PROGRAMA,2.640583
4,AEROLINEA,0.539006
5,MES_PARTIDA,0.210461
6,AEROPUERTO_DESTINO,0.172964
7,DISTANCIA,0.084732
8,AEROPUERTO_ORIGEN,0.071130
9,TEMPORADA,0.036600


In [ ]:
print("Mejor iteración:", at_model.get_best_iteration())
print("Mejor score:", at_model.get_best_score())

Mejor iteración: 999
Mejor score: {'learn': {'Logloss': 0.21573356285860207}, 'validation': {'Logloss': 0.2175249293483713}}



**Nota:** Logloss (también llamado logistic loss o cross‑entropy loss) es una métrica que mide qué tan bien un modelo de clasificación que entrega probabilidades se ajusta a las etiquetas reales. Cuanto más bajo es el logloss, mejor está calibrado el modelo.
